# Pergam Ground Coordinate Processing

This notebook processes Pergam drone sensor files (`.csv` and `.xlsx`), detects the airborne interval,
corrects per-flight GPS drift, computes laser ground-intersection coordinates, and provides quick methane
visualizations.

Use the configuration cell below to set input/output directories, detector thresholds, label policy, and
plot-saving behavior. The processing pipeline writes `-results.csv` files, and the validation cell summarizes
whether the detector and output policies behaved as expected.


In [ ]:
import glob
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import cm
from matplotlib.colors import Normalize


# Configuration

Set these values before running the notebook. The defaults keep CSV `Names of Flights` empty for unlabeled
in-flight rows, blank out ground-coordinate outputs for `On Ground` rows, and avoid auto-saving plots.


In [ ]:
INPUT_DIR = "./data/3_13_2026_Vernal_Cooked"
OUTPUT_DIR = None  # None -> same directory as INPUT_DIR

# Flight detection
FLIGHT_ALT_COL = "Altitude"
TAKEOFF_THRESHOLD_M = 0.75
LANDING_THRESHOLD_M = 0.35
ALTITUDE_SMOOTHING_WINDOW = 30
MIN_TAKEOFF_RUN = 5
MIN_LANDING_RUN = 5
DRIFT_AVG_POINTS = 10

# Label policy
ON_GROUND_LABEL = "On Ground"
PRE_LABEL_VALUE = "Pre-label"
CSV_UNLABELED_POLICY = "empty"  # "empty" or "pre-label"

# Ground-coordinate policy
EMIT_GROUND_COORDS_ON_GROUND = False
LIDAR_BLOCKED_THRESHOLD_M = 0.05
LIDAR_SATURATION_M = 250.0
MIN_DOWN_COMPONENT = 0.01

# Plotting
SAVE_PLOTS = False
PLOT_OUTPUT_DIR = None  # None -> OUTPUT_DIR or INPUT_DIR
FILE_TO_PLOT = None  # None -> first processed file
METHANE_CLIP_MAX = 100


# Step 1 - Unified File Reader

`read_sensor_file()` standardizes Pergam `.csv` and `.xlsx` inputs into one dataframe shape.
XLSX files provide a session description and segment labels. CSV files do not, so the notebook inserts a
`Names of Flights` column and rebuilds timing later from the `Time` column when needed.


In [ ]:
def read_sensor_file(path):
    """
    Read a sensor data file (.csv or .xlsx) and return a standardized DataFrame.

    Returns
    -------
    (df, session_description) : tuple
        df : pd.DataFrame with standardized columns
        session_description : str (from xlsx cell A1, or filename for csv)
    """
    path = Path(path)
    ext = path.suffix.lower()

    if ext == ".xlsx":
        df_header = pd.read_excel(path, header=None, nrows=1)
        session_description = str(df_header.iloc[0, 0])

        df_raw = pd.read_excel(path, header=None, skiprows=1)
        col_names = df_raw.iloc[0].tolist()
        df = df_raw.iloc[1:].reset_index(drop=True)
        df.columns = col_names

        first_col = df.columns[0]
        if pd.isna(first_col) or str(first_col).strip() == "":
            df.rename(columns={first_col: "Names of Flights"}, inplace=True)

    elif ext == ".csv":
        df = pd.read_csv(path)
        session_description = path.name

        if "Names of Flights" not in df.columns:
            df.insert(0, "Names of Flights", pd.Series([pd.NA] * len(df), dtype="object"))

        drop_cols = ["ALT:ID"]
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

    else:
        raise ValueError(f"Unsupported file type: {ext}")

    numeric_cols = [
        "Elapsed", "Pitch", "Roll", "Heading",
        "Latitude", "Longitude", "Altitude",
        "Velocity", "ALT:Altitude", "GAS:Methane", "GAS:Status",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df.attrs["source_ext"] = ext
    df.attrs["session_description"] = session_description

    print(f"  Loaded {path}: {len(df)} rows, session='{session_description}'")
    return df, session_description


# Step 1b - Normalize Time Column

CSV `Elapsed` values are often destroyed by Excel scientific notation export, while XLSX files retain usable
epoch timestamps. `normalize_elapsed()` rebuilds a monotonic `Elapsed` column from `Time` when necessary so
flight detection and drift correction can use one consistent time axis.


In [ ]:
def normalize_elapsed(df):
    """
    Ensure the Elapsed column contains reliable, monotonically increasing values.

    Handles three cases:
      1. XLSX Elapsed: epoch ms with full precision (> 1 unique value) - use as-is
      2. CSV Elapsed: destroyed by scientific notation (1 unique value) - rebuild from Time
      3. Fallback: use row index as a proxy

    Returns
    -------
    pd.DataFrame with a reliable monotonic 'Elapsed' column (float, milliseconds).
    """
    elapsed = pd.to_numeric(df.get("Elapsed"), errors="coerce")
    unique_vals = elapsed.dropna().unique()
    if len(unique_vals) > 1:
        df["Elapsed"] = elapsed
        return df

    if "Time" not in df.columns:
        print("  WARNING: No usable Elapsed or Time column; using row index")
        df["Elapsed"] = df.index.astype(float)
        return df

    time_col = df["Time"]
    sample = time_col.dropna().iloc[0] if time_col.notna().any() else None
    if sample is None:
        print("  WARNING: Time column is all NaN; using row index")
        df["Elapsed"] = df.index.astype(float)
        return df

    import datetime

    if isinstance(sample, datetime.time):
        def time_to_ms(t):
            if pd.isna(t):
                return np.nan
            return (t.hour * 3600 + t.minute * 60 + t.second) * 1000 + t.microsecond / 1000

        df["Elapsed"] = time_col.apply(time_to_ms)
        print("  Elapsed rebuilt from Time (datetime.time -> ms since midnight)")

    elif isinstance(sample, str) and ":" in sample:
        def mmss_to_ms(val):
            if pd.isna(val):
                return np.nan
            s = str(val).strip()
            if not s:
                return np.nan
            parts = s.split(":")
            try:
                if len(parts) == 2:
                    return (float(parts[0]) * 60 + float(parts[1])) * 1000
                if len(parts) == 3:
                    return (float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])) * 1000
                return float(s) * 1000
            except ValueError:
                return np.nan

        df["Elapsed"] = time_col.apply(mmss_to_ms)
        print("  Elapsed rebuilt from Time (MM:SS.s -> ms)")

    else:
        df["Elapsed"] = pd.to_numeric(time_col, errors="coerce")
        if df["Elapsed"].notna().sum() == 0:
            print("  WARNING: Could not parse Time column; using row index")
            df["Elapsed"] = df.index.astype(float)
        else:
            print("  Elapsed rebuilt from Time (numeric)")

    return df


# Step 2 - Flight Boundary Detection

The notebook now uses positive GPS altitude with hysteresis instead of absolute altitude.
A flight starts after a sustained run above the takeoff threshold and ends at the first sustained run below the
landing threshold. This prevents post-flight negative drift from being misread as a second takeoff.


In [ ]:
def detect_flights(
    df,
    alt_col="Altitude",
    takeoff_threshold=0.75,
    landing_threshold=0.35,
    window=30,
    min_takeoff_run=5,
    min_landing_run=5,
    on_ground_label="On Ground",
):
    """
    Detect flight boundaries using positive GPS altitude with hysteresis.

    Returns half-open flight intervals: (start_row, end_row), where end_row is the first
    on-ground row after landing and is excluded from the flight span.
    """
    alt = pd.to_numeric(df[alt_col], errors="coerce")
    smooth = alt.rolling(window=window, center=True, min_periods=1).mean()

    flights = []
    takeoff_rows = []
    landing_rows = []
    in_flight = False
    flight_start = None
    i = 0
    n_rows = len(df)

    while i < n_rows:
        value = smooth.iloc[i]
        if pd.isna(value):
            i += 1
            continue

        if not in_flight and value >= takeoff_threshold:
            run_end = i
            while run_end < n_rows and pd.notna(smooth.iloc[run_end]) and smooth.iloc[run_end] >= takeoff_threshold:
                run_end += 1
            if run_end - i >= min_takeoff_run:
                flight_start = i
                takeoff_rows.append(i)
                in_flight = True
                i = run_end
                continue
            i = run_end
            continue

        if in_flight and value <= landing_threshold:
            run_end = i
            while run_end < n_rows and pd.notna(smooth.iloc[run_end]) and smooth.iloc[run_end] <= landing_threshold:
                run_end += 1
            if run_end - i >= min_landing_run:
                landing_rows.append(i)
                flights.append((flight_start, i))
                in_flight = False
                flight_start = None
                i = run_end
                continue
            i = run_end
            continue

        i += 1

    if in_flight and flight_start is not None:
        flights.append((flight_start, n_rows))

    df["Flight"] = on_ground_label
    for flight_num, (start, end) in enumerate(flights, 1):
        if end > start:
            df.loc[df.index[start:end], "Flight"] = f"Flight {flight_num}"

    df.attrs["flight_detection"] = {
        "takeoff_rows": takeoff_rows,
        "landing_rows": landing_rows,
        "takeoff_threshold": takeoff_threshold,
        "landing_threshold": landing_threshold,
        "window": window,
        "min_takeoff_run": min_takeoff_run,
        "min_landing_run": min_landing_run,
    }

    print(f"  Detected {len(flights)} flight(s)")
    for flight_num, (start, end) in enumerate(flights, 1):
        max_alt = alt.iloc[start:end].max()
        print(f"    Flight {flight_num}: rows {start}-{end - 1} ({end - start} samples), max alt={max_alt:.1f}m")

    return df, flights


# Step 3 - Fill Flight Labels

XLSX segment labels are forward-filled within each detected flight. CSV files do not contain native segment labels,
so unlabeled in-flight rows are left empty by default. On-ground rows are labeled `On Ground` explicitly.


In [ ]:
def fill_flight_labels(
    df,
    on_ground_label="On Ground",
    pre_label_value="Pre-label",
    csv_unlabeled_policy="empty",
):
    """
    Forward-fill the 'Names of Flights' column while preserving the chosen CSV policy.
    """
    col = "Names of Flights"
    if col not in df.columns:
        df[col] = pd.Series([pd.NA] * len(df), dtype="object")

    df[col] = df[col].astype("object")
    df[col] = df[col].apply(
        lambda value: value if pd.notna(value) and str(value).strip() != "" else pd.NA
    )

    for flight_name in df["Flight"].dropna().unique():
        mask = df["Flight"] == flight_name
        if flight_name == on_ground_label:
            df.loc[mask, col] = on_ground_label
            continue

        flight_labels = df.loc[mask, col].copy()
        has_native_labels = flight_labels.notna().any()
        filled = flight_labels.ffill()

        if has_native_labels or csv_unlabeled_policy == "pre-label":
            filled = filled.fillna(pre_label_value)
        else:
            filled = filled.fillna("")

        df.loc[mask, col] = filled.astype("object")

    return df


# Step 4 - Per-Flight 3D GPS Drift Correction

Drift correction now consumes the same half-open flight intervals returned by `detect_flights()`.
This keeps drift interpolation aligned with the corrected flight boundaries and avoids using the first on-ground row
after landing in the in-flight correction range.


In [ ]:
def correct_drift_per_flight(df, flights, n_avg=10):
    """
    Apply linear 3D GPS drift correction independently per flight.
    """
    df["Latitude_adj"] = df["Latitude"].copy()
    df["Longitude_adj"] = df["Longitude"].copy()
    df["Altitude_adj"] = df["Altitude"].copy()

    drift_summaries = []

    for flight_num, (start, end) in enumerate(flights, 1):
        idx = df.index[start:end]
        flight = df.loc[idx].copy()

        valid_mask = (
            flight["Latitude"].notna()
            & flight["Longitude"].notna()
            & flight["Altitude"].notna()
            & flight["Elapsed"].notna()
        )
        valid = flight.loc[valid_mask]
        if len(valid) < 2:
            print(f"    Flight {flight_num}: too few valid points for drift correction")
            continue

        n_edge = min(n_avg, len(valid) // 2)
        if n_edge == 0:
            print(f"    Flight {flight_num}: too few valid points for drift correction")
            continue

        start_mean = valid.head(n_edge)[["Latitude", "Longitude", "Altitude"]].mean()
        end_mean = valid.tail(n_edge)[["Latitude", "Longitude", "Altitude"]].mean()

        dlat = end_mean["Latitude"] - start_mean["Latitude"]
        dlon = end_mean["Longitude"] - start_mean["Longitude"]
        dalt = end_mean["Altitude"] - start_mean["Altitude"]

        elapsed = valid["Elapsed"]
        t0 = elapsed.iloc[0]
        tN = elapsed.iloc[-1]
        if tN == t0:
            print(f"    Flight {flight_num}: elapsed time is constant; skipping drift correction")
            continue

        frac = (flight["Elapsed"] - t0) / (tN - t0)
        frac = frac.clip(lower=0, upper=1)

        df.loc[idx, "Latitude_adj"] = flight["Latitude"] - frac * dlat
        df.loc[idx, "Longitude_adj"] = flight["Longitude"] - frac * dlon
        df.loc[idx, "Altitude_adj"] = flight["Altitude"] - frac * dalt

        drift_m = np.sqrt(
            (dlat * 111320) ** 2
            + (dlon * 111320 * np.cos(np.radians(start_mean["Latitude"]))) ** 2
        )
        drift_summaries.append(
            {
                "flight": flight_num,
                "horizontal_drift_m": float(drift_m),
                "altitude_drift_m": float(dalt),
            }
        )
        print(
            f"    Flight {flight_num}: rows {start}-{end - 1}, "
            f"horizontal drift={drift_m:.2f}m, altitude drift={dalt:+.3f}m"
        )

    df.attrs["drift_summaries"] = drift_summaries
    return df


# Step 5 - Compute Ground Coordinates

Ground coordinates are computed from the drift-corrected drone pose and the corrected ray-ground intersection
math. By default, this notebook now leaves `ground_*` outputs empty for `On Ground` rows so the output files only
expose laser intersections for airborne samples.


In [ ]:
def compute_ground_coordinates(
    df,
    emit_on_ground=False,
    on_ground_label="On Ground",
    lidar_blocked_threshold=0.05,
    lidar_saturation=250.0,
    min_down_component=0.01,
):
    """
    Compute laser ground-intersection coordinates from drone position and orientation.
    """
    df["ground_lat"] = np.nan
    df["ground_lon"] = np.nan
    df["ground_alt_est"] = np.nan
    df["horizontal_offset_m"] = np.nan

    lat_col = "Latitude_adj" if "Latitude_adj" in df.columns else "Latitude"
    lon_col = "Longitude_adj" if "Longitude_adj" in df.columns else "Longitude"
    alt_col = "Altitude_adj" if "Altitude_adj" in df.columns else "Altitude"

    required = [lat_col, lon_col, alt_col, "Heading", "Pitch", "Roll", "ALT:Altitude"]
    for col in required:
        if col not in df.columns:
            print(f"  WARNING: Missing column '{col}', skipping ground computation")
            return df

    valid = df[required].notna().all(axis=1)
    if not emit_on_ground and "Flight" in df.columns:
        valid &= df["Flight"] != on_ground_label

    lidar_alt = df["ALT:Altitude"]
    if "Flight" in df.columns:
        in_flight = df["Flight"] != on_ground_label
    else:
        in_flight = pd.Series(True, index=df.index)

    lidar_blocked = (lidar_alt <= lidar_blocked_threshold) & in_flight
    lidar_saturated = lidar_alt >= lidar_saturation
    valid = valid & ~lidar_blocked & ~lidar_saturated

    if lidar_blocked.any():
        print(f"  Filtered {int(lidar_blocked.sum())} rows: LIDAR blocked (reading near zero in flight)")
    if lidar_saturated.any():
        print(f"  Filtered {int(lidar_saturated.sum())} rows: LIDAR saturated ({lidar_saturation:.0f}m)")

    if valid.sum() == 0:
        print("  WARNING: No valid rows for ground computation")
        return df

    idx = df.index[valid]
    lat = np.radians(df.loc[idx, lat_col].values)
    lon = np.radians(df.loc[idx, lon_col].values)
    yaw = np.radians(df.loc[idx, "Heading"].values)
    pitch = np.radians(df.loc[idx, "Pitch"].values)
    roll = np.radians(df.loc[idx, "Roll"].values)
    s = df.loc[idx, "ALT:Altitude"].values
    gps_alt = df.loc[idx, alt_col].values

    a = 6378137.0
    f = 1 / 298.257223563
    e2 = 1 - (1 - f) ** 2

    cy, sy = np.cos(yaw), np.sin(yaw)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cr, sr = np.cos(roll), np.sin(roll)

    wn = cy * sp * cr + sy * sr
    we = sy * sp * cr - cy * sr
    wd = cp * cr

    bad_wd = wd <= min_down_component
    if bad_wd.any():
        print(f"  Filtered {int(bad_wd.sum())} rows: laser direction not pointing down")
        wn = wn[~bad_wd]
        we = we[~bad_wd]
        wd = wd[~bad_wd]
        s = s[~bad_wd]
        gps_alt = gps_alt[~bad_wd]
        lat = lat[~bad_wd]
        lon = lon[~bad_wd]
        idx = idx[~bad_wd]

    if len(idx) == 0:
        print("  WARNING: No valid rows remain after downward-direction filter")
        return df

    t = s / wd
    dn = t * wn
    de = t * we

    sinlat = np.sin(lat)
    Rn = a / np.sqrt(1 - e2 * sinlat**2)
    Rm = a * (1 - e2) / (1 - e2 * sinlat**2) ** 1.5

    dlat_rad = dn / Rm
    dlon_rad = de / (Rn * np.cos(lat))

    df.loc[idx, "ground_lat"] = np.degrees(lat + dlat_rad)
    df.loc[idx, "ground_lon"] = np.degrees(lon + dlon_rad)
    df.loc[idx, "ground_alt_est"] = gps_alt - s
    df.loc[idx, "horizontal_offset_m"] = np.sqrt(dn**2 + de**2)

    print(f"  Ground coordinates computed for {len(idx)}/{len(df)} rows")
    return df


# Step 6 - Process Directory

`process_directory()` wires the full pipeline together, writes `-results.csv` outputs, and records per-file
processing metadata for the validation step below.


In [ ]:
def process_directory(
    input_dir,
    output_dir=None,
    *,
    alt_col="Altitude",
    takeoff_threshold=0.75,
    landing_threshold=0.35,
    window=30,
    min_takeoff_run=5,
    min_landing_run=5,
    n_avg=10,
    on_ground_label="On Ground",
    pre_label_value="Pre-label",
    csv_unlabeled_policy="empty",
    emit_ground_coords_on_ground=False,
    lidar_blocked_threshold=0.05,
    lidar_saturation=250.0,
    min_down_component=0.01,
):
    """
    Process all .csv and .xlsx sensor files in a directory.
    """
    input_dir = Path(input_dir)
    output_dir = input_dir if output_dir is None else Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    all_files = sorted(list(input_dir.glob("*.csv")) + list(input_dir.glob("*.xlsx")))
    files = [path for path in all_files if not path.name.endswith("-results.csv")]
    if not files:
        print(f"No .csv or .xlsx files found in {input_dir}")
        return {}

    results = {}
    print(f"Processing {len(files)} file(s) from {input_dir}\n")

    for filepath in files:
        print(f"=== {filepath.name} ===")
        df, session_desc = read_sensor_file(filepath)
        df = normalize_elapsed(df)
        df, flights = detect_flights(
            df,
            alt_col=alt_col,
            takeoff_threshold=takeoff_threshold,
            landing_threshold=landing_threshold,
            window=window,
            min_takeoff_run=min_takeoff_run,
            min_landing_run=min_landing_run,
            on_ground_label=on_ground_label,
        )
        df = fill_flight_labels(
            df,
            on_ground_label=on_ground_label,
            pre_label_value=pre_label_value,
            csv_unlabeled_policy=csv_unlabeled_policy,
        )
        df = correct_drift_per_flight(df, flights, n_avg=n_avg)
        df = compute_ground_coordinates(
            df,
            emit_on_ground=emit_ground_coords_on_ground,
            on_ground_label=on_ground_label,
            lidar_blocked_threshold=lidar_blocked_threshold,
            lidar_saturation=lidar_saturation,
            min_down_component=min_down_component,
        )

        on_ground_mask = df["Flight"] == on_ground_label
        summary = {
            "source_path": str(filepath),
            "session_description": session_desc,
            "flights": flights,
            "on_ground_ground_rows": int((on_ground_mask & df["ground_lat"].notna()).sum()),
            "in_flight_ground_rows": int((~on_ground_mask & df["ground_lat"].notna()).sum()),
        }
        df.attrs["processing_summary"] = summary

        out_path = output_dir / f"{filepath.stem}-results.csv"
        df.to_csv(out_path, index=False)
        print(f"  Output written to: {out_path}")
        print(
            f"  Summary: flights={len(flights)}, "
            f"in-flight ground rows={summary['in_flight_ground_rows']}, "
            f"on-ground ground rows={summary['on_ground_ground_rows']}\n"
        )

        results[filepath.name] = df

    print(f"Done. Processed {len(results)} file(s).")
    return results


# Validation Helpers

Run this after processing to confirm that the updated detector and output policies behaved as intended.
The summary checks flight count, landing boundaries, on-ground output behavior, and the chosen CSV label policy.


In [ ]:
def validate_processed_results(
    results,
    *,
    on_ground_label="On Ground",
    csv_unlabeled_policy="empty",
    emit_ground_coords_on_ground=False,
    pre_label_value="Pre-label",
    expected_flights=1,
):
    """
    Build a compact per-file validation summary for the processed outputs.
    """
    rows = []
    for name, df in results.items():
        summary = df.attrs.get("processing_summary", {})
        flights = summary.get("flights", [])
        source_ext = df.attrs.get("source_ext", Path(name).suffix.lower())

        if "Flight" in df.columns:
            on_ground_mask = df["Flight"] == on_ground_label
        else:
            on_ground_mask = pd.Series(False, index=df.index)
        in_flight_mask = ~on_ground_mask

        landing_boundary_ok = all(
            end >= len(df) or df.iloc[end]["Flight"] == on_ground_label
            for start, end in flights
        ) if flights else False

        on_ground_ground_rows = int((on_ground_mask & df["ground_lat"].notna()).sum()) if "ground_lat" in df.columns else 0
        in_flight_ground_rows = int((in_flight_mask & df["ground_lat"].notna()).sum()) if "ground_lat" in df.columns else 0

        csv_empty_inflight = np.nan
        csv_prelabel_inflight = np.nan
        csv_label_policy_ok = True
        if source_ext == ".csv" and "Names of Flights" in df.columns:
            labels = df["Names of Flights"].fillna("")
            csv_empty_inflight = int((in_flight_mask & labels.eq("")).sum())
            csv_prelabel_inflight = int((in_flight_mask & labels.eq(pre_label_value)).sum())
            if csv_unlabeled_policy == "empty":
                csv_label_policy_ok = csv_prelabel_inflight == 0
            else:
                csv_label_policy_ok = csv_empty_inflight == 0

        on_ground_policy_ok = on_ground_ground_rows == 0 if not emit_ground_coords_on_ground else True
        flight_count_ok = len(flights) == expected_flights

        rows.append(
            {
                "file": name,
                "flights_detected": len(flights),
                "flight_count_ok": flight_count_ok,
                "landing_boundary_ok": landing_boundary_ok,
                "on_ground_ground_rows": on_ground_ground_rows,
                "on_ground_policy_ok": on_ground_policy_ok,
                "in_flight_ground_rows": in_flight_ground_rows,
                "csv_empty_inflight": csv_empty_inflight,
                "csv_prelabel_inflight": csv_prelabel_inflight,
                "csv_label_policy_ok": csv_label_policy_ok,
            }
        )

    validation_df = pd.DataFrame(rows)
    if validation_df.empty:
        print("No processed files available for validation.")
        return validation_df

    validation_df["all_checks_pass"] = validation_df[
        ["flight_count_ok", "landing_boundary_ok", "on_ground_policy_ok", "csv_label_policy_ok"]
    ].all(axis=1)

    print("Validation summary:")
    print(validation_df.to_string(index=False))
    return validation_df


# Visualization Functions

These helpers summarize methane values and plot only the processed dataframe you choose later in the notebook.
Plot saving is opt-in and uses file-specific names when enabled.


In [ ]:
def _maybe_save_figure(fig, plot_name, save_plot=False, plot_dir=None, file_stem=None):
    """Save a figure only when explicitly requested."""
    if not save_plot:
        return None

    if plot_dir is None:
        raise ValueError("plot_dir is required when save_plot=True")

    plot_dir = Path(plot_dir)
    plot_dir.mkdir(parents=True, exist_ok=True)
    prefix = f"{file_stem}-" if file_stem else ""
    out_path = plot_dir / f"{prefix}{plot_name}.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved plot: {out_path}")
    return out_path


def summarize_methane(ground_df, column="GAS:Methane", clip_max=None):
    """Summarize methane values from the processed ground dataframe."""
    if column not in ground_df.columns:
        raise ValueError(f"Column '{column}' not found in dataframe.")

    s = pd.to_numeric(ground_df[column], errors="coerce").dropna()
    if clip_max is not None:
        s = s.clip(upper=clip_max)

    stats = {
        "count": int(s.size),
        "min": float(s.min()),
        "p01": float(np.percentile(s, 1)),
        "p25": float(np.percentile(s, 25)),
        "median": float(np.median(s)),
        "mean": float(s.mean()),
        "p75": float(np.percentile(s, 75)),
        "p99": float(np.percentile(s, 99)),
        "max": float(s.max()),
        "std": float(s.std(ddof=1)),
    }
    print(f"\n=== Methane summary (clip_max={clip_max}) ===")
    for key, value in stats.items():
        print(f"{key:>6}: {value:.3f}")
    return stats


def plot_methane_hist_seaborn(
    ground_df,
    column="GAS:Methane",
    clip_max=None,
    kde=True,
    bins="auto",
    color="steelblue",
    save_plot=False,
    plot_dir=None,
    file_stem=None,
):
    """Plot a methane histogram with an optional KDE overlay."""
    if column not in ground_df.columns:
        raise ValueError(f"Column '{column}' not found in dataframe.")

    s = pd.to_numeric(ground_df[column], errors="coerce").dropna()
    above = (s > clip_max).sum() if clip_max is not None else 0
    s = s.clip(upper=clip_max) if clip_max is not None else s

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(s, kde=kde, bins=bins, color=color, edgecolor="black", alpha=0.7, ax=ax)
    ax.set_xlabel("Methane concentration")
    ax.set_ylabel("Frequency")
    ax.set_title(f"Methane Histogram (clip_max={clip_max}, KDE={kde})")
    if clip_max is not None and above > 0:
        ax.annotate(
            f"{above} values > {clip_max} capped",
            xy=(0.98, 0.95),
            xycoords="axes fraction",
            ha="right",
            va="top",
            color="red",
            fontsize=9,
        )
    ax.grid(alpha=0.3)
    fig.tight_layout()
    _maybe_save_figure(fig, "methane_hist_seaborn", save_plot, plot_dir, file_stem)
    plt.show()


def plot_methane_hist_box_notched(
    ground_df,
    column="GAS:Methane",
    clip_max=None,
    kde=True,
    bins="auto",
    color="steelblue",
    save_plot=False,
    plot_dir=None,
    file_stem=None,
):
    """Plot a notched boxplot above a histogram and optional KDE."""
    if column not in ground_df.columns:
        raise ValueError(f"Column '{column}' not found in dataframe.")

    s = pd.to_numeric(ground_df[column], errors="coerce").dropna()
    above = (s > clip_max).sum() if clip_max is not None else 0
    s = s.clip(upper=clip_max) if clip_max is not None else s

    fig, (ax_box, ax_hist) = plt.subplots(
        2,
        1,
        figsize=(9, 6),
        gridspec_kw={"height_ratios": [0.22, 0.78], "hspace": 0.06},
        sharex=True,
    )

    ax_box.boxplot(
        s.values,
        vert=False,
        notch=True,
        patch_artist=True,
        flierprops=dict(markersize=3, marker="o", markerfacecolor="black", markeredgecolor="black", alpha=0.6),
        boxprops=dict(facecolor=color, edgecolor="black"),
        medianprops=dict(color="black", linewidth=1.5),
        whiskerprops=dict(color="black"),
        capprops=dict(color="black"),
    )
    ax_box.set_yticks([])
    ax_box.set_ylabel("")
    ax_box.set_xlabel("")
    ax_box.set_title(f"Methane Distribution (clip_max={clip_max}, KDE={kde})")

    sns.histplot(s, kde=kde, bins=bins, color=color, edgecolor="black", alpha=0.7, ax=ax_hist)
    ax_hist.set_xlabel("Methane concentration")
    ax_hist.set_ylabel("Frequency")
    if clip_max is not None and above > 0:
        ax_hist.annotate(
            f"{above} values > {clip_max} capped",
            xy=(0.98, 0.95),
            xycoords="axes fraction",
            ha="right",
            va="top",
            color="red",
            fontsize=9,
        )

    fig.tight_layout()
    _maybe_save_figure(fig, "methane_hist_box_notched", save_plot, plot_dir, file_stem)
    plt.show()


# Run Processing Pipeline

Execute the full pipeline here. The validation summary reports whether each file produced the expected single
flight, respected the landing boundary convention, and followed the chosen output policies.


In [ ]:
results = process_directory(
    INPUT_DIR,
    OUTPUT_DIR,
    alt_col=FLIGHT_ALT_COL,
    takeoff_threshold=TAKEOFF_THRESHOLD_M,
    landing_threshold=LANDING_THRESHOLD_M,
    window=ALTITUDE_SMOOTHING_WINDOW,
    min_takeoff_run=MIN_TAKEOFF_RUN,
    min_landing_run=MIN_LANDING_RUN,
    n_avg=DRIFT_AVG_POINTS,
    on_ground_label=ON_GROUND_LABEL,
    pre_label_value=PRE_LABEL_VALUE,
    csv_unlabeled_policy=CSV_UNLABELED_POLICY,
    emit_ground_coords_on_ground=EMIT_GROUND_COORDS_ON_GROUND,
    lidar_blocked_threshold=LIDAR_BLOCKED_THRESHOLD_M,
    lidar_saturation=LIDAR_SATURATION_M,
    min_down_component=MIN_DOWN_COMPONENT,
)

validation_df = validate_processed_results(
    results,
    on_ground_label=ON_GROUND_LABEL,
    csv_unlabeled_policy=CSV_UNLABELED_POLICY,
    emit_ground_coords_on_ground=EMIT_GROUND_COORDS_ON_GROUND,
    pre_label_value=PRE_LABEL_VALUE,
    expected_flights=1,
)


# Map of Flight and Measurements

Pick one processed file, filter to in-flight rows with valid ground coordinates, and plot the ground-intersection
points alongside the corrected drone positions.


In [ ]:
result_files = list(results.keys())
print("Available files:", result_files)

selected_name = FILE_TO_PLOT if FILE_TO_PLOT in results else (result_files[0] if result_files else None)
ground_df = results.get(selected_name) if selected_name else None
plot_df = None
plot_file_stem = None

if ground_df is not None:
    plot_file_stem = Path(selected_name).stem
    print(f"\nVisualizing: {selected_name}")
    plot_df = ground_df[
        (ground_df["Flight"] != ON_GROUND_LABEL)
        & ground_df["ground_lat"].notna()
    ].copy()

    plt.figure(figsize=(20, 8))
    ax = sns.scatterplot(
        data=plot_df,
        x="ground_lon",
        y="ground_lat",
        hue="GAS:Methane",
        palette="viridis",
        size="GAS:Methane",
        sizes=(20, 200),
        hue_norm=(0, 100),
    )

    sns.scatterplot(
        data=plot_df,
        x="Longitude_adj",
        y="Latitude_adj",
        color="black",
        s=5,
        label="Drone Loc",
        marker="x",
        ax=ax,
    )

    plt.title("Ground Coordinates (Colored by Methane) and Drone Locations")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")

    cmap = plt.get_cmap("viridis")
    norm = Normalize(vmin=0, vmax=100)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="GAS:Methane (Capped at 100 ppm-m)")

    plt.grid(True)
    plt.legend(loc="lower left", fontsize=9)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.tight_layout()
    plt.show()
else:
    print("No processed file available for visualization.")


# Methane Visualization Plots

These plots operate on the selected file's in-flight rows with valid ground coordinates.
Plot saving remains disabled unless `SAVE_PLOTS = True` in the configuration cell.


In [ ]:
plot_dir = PLOT_OUTPUT_DIR or OUTPUT_DIR or INPUT_DIR

if plot_df is not None and not plot_df.empty:
    summarize_methane(plot_df, clip_max=METHANE_CLIP_MAX)
    plot_methane_hist_seaborn(
        plot_df,
        clip_max=METHANE_CLIP_MAX,
        kde=True,
        save_plot=SAVE_PLOTS,
        plot_dir=plot_dir,
        file_stem=plot_file_stem,
    )
    plot_methane_hist_box_notched(
        plot_df,
        column="GAS:Methane",
        clip_max=METHANE_CLIP_MAX,
        kde=True,
        bins=50,
        color="teal",
        save_plot=SAVE_PLOTS,
        plot_dir=plot_dir,
        file_stem=plot_file_stem,
    )
else:
    print("No in-flight ground-coordinate rows are available for methane visualization.")
